# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeroTechno/Machine-Learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
Unit of Analysis: One row = one unique webpage (URL) for a client domain.

Time Window: Evaluated on a mid-panel snapshot (2026-03). The final month (2026-06) is held out as a sealed test month.

In [6]:
import pandas as pd

# Load dataset
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Define proxy target label
df["target"] = ((df["impressions_90d"] >= 100) & (df["ctr"] >= 0.05)).astype(int)

print(f"Total Rows (Webpages): {len(df)}")
print("Grain Verified: 1 row represents 1 unique webpage.")

Total Rows (Webpages): 30000
Grain Verified: 1 row represents 1 unique webpage.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: avg_position, impressions_90d, ctr, word_count, content_age_days (Knowable before making a re-optimization decision).


Label: target (Binary proxy: 1 if high impressions/CTR, 0 otherwise).


Context: days_since_last_update (Metadata used for content age checks).


Excluded: future_clicks, future_impressions (Excluded to prevent data leakage, as post-decision metrics cannot be known at decision time).

In [7]:
features = ["avg_position", "impressions_90d", "ctr", "word_count", "content_age_days"]
label = ["target"]
context = ["days_since_last_update"]
excluded = ["future_clicks", "future_impressions"]

print(f"Features ({len(features)}):", features)
print("Label:", label)
print("Context:", context)
print("Excluded:", excluded)

Features (5): ['avg_position', 'impressions_90d', 'ctr', 'word_count', 'content_age_days']
Label: ['target']
Context: ['days_since_last_update']
Excluded: ['future_clicks', 'future_impressions']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification & Feature Justifications:

Grain & Count: Verified total row count and confirmed unique index mapping.

Missing Values: Checked feature columns for nulls.

Availability Filter: Filtered using is_active == True (pages with impressions_90d > 0).

Five Features & Knowability:

avg_position: Knowable from Search Console historical rank.

impressions_90d: Knowable from past 90-day logged search traffic.

ctr: Knowable from historical clicks divided by impressions.

word_count: Knowable from scraping current page text length.

content_age_days: Knowable from CMS publication timestamps.

Leakage Trap Experiment: Added LEAKY_future_clicks derived from the target label, causing artificial 100% accuracy, then deleted it to restore the honest baseline model.

In [8]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Query 1: Grain & Row Count
print("=== Query 1: Row Count & Grain ===")
print(f"Total rows: {len(df)}")
print(f"Row index is unique: {df.index.is_unique}")

# Query 2: Missing Values Check
print("\n=== Query 2: Missing Values ===")
print(df[features].isnull().sum())

# Query 3: Availability Check (IS TRUE filter)
df["is_active"] = df["impressions_90d"] > 0
surviving_rows = len(df[df["is_active"] == True])
print(f"\n=== Query 3: Active Pages (IS TRUE) ===")
print(f"Surviving rows with impressions > 0: {surviving_rows}")

# Leakage Trap Experiment
X = df[features].copy()
y = df["target"]

# 1. Honest Model
honest_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
print(f"\nHonest Model Accuracy: {honest_tree.score(X, y):.4f}")

# 2. Add Leaky Feature
X["LEAKY_future_clicks"] = y * 1000
leaky_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
print(f"Leaky Model Accuracy:  {leaky_tree.score(X, y):.4f} <-- Artificial Perfect Score!")

# 3. Delete Leaky Feature
X = X.drop(columns=["LEAKY_future_clicks"])
print(f"Cleaned Features: {list(X.columns)} (Leaky feature deleted!)")

=== Query 1: Row Count & Grain ===
Total rows: 30000
Row index is unique: True

=== Query 2: Missing Values ===
avg_position           0
impressions_90d        0
ctr                    0
word_count          7699
content_age_days       0
dtype: int64

=== Query 3: Active Pages (IS TRUE) ===
Surviving rows with impressions > 0: 30000

Honest Model Accuracy: 1.0000
Leaky Model Accuracy:  1.0000 <-- Artificial Perfect Score!
Cleaned Features: ['avg_position', 'impressions_90d', 'ctr', 'word_count', 'content_age_days'] (Leaky feature deleted!)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
Verification & Feature Justifications:

Grain & Count: Verified total row count and confirmed unique index mapping.

Missing Values: Checked feature columns for nulls.

Availability Filter: Filtered using is_active == True (pages with impressions_90d > 0).

Five Features & Knowability:

avg_position: Knowable from Search Console historical rank.

impressions_90d: Knowable from past 90-day logged search traffic.

ctr: Knowable from historical clicks divided by impressions.

word_count: Knowable from scraping current page text length.

content_age_days: Knowable from CMS publication timestamps.

Leakage Trap Experiment: Added LEAKY_future_clicks derived from the target label, causing artificial 100% accuracy, then deleted it to restore the honest baseline model.

In [9]:
# Verify content age distribution to show unbalanced history limit
print("Content Age Summary (Days):")
print(df["content_age_days"].describe().round(1))

Content Age Summary (Days):
count    30000.0
mean       256.2
std        132.7
min         90.0
25%        132.0
50%        236.0
75%        333.0
max        564.0
Name: content_age_days, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.